# Bài Lab Thực hành: Hồi quy tuyến tính

Chào mừng bạn đến với bài lab thực hành đầu tiên! Trong lab này, bạn sẽ triển khai hồi quy tuyến tính với một biến để dự đoán lợi nhuận cho một chuỗi nhà hàng.


# Dàn ý
- [ 1 - Các gói thư viện ](#1)
- [ 2 - Hồi quy tuyến tính với một biến ](#2)
  - [ 2.1 Phát biểu bài toán](#2.1)
  - [ 2.2  Tập dữ liệu](#2.2)
  - [ 2.3 Ôn lại về hồi quy tuyến tính](#2.3)
  - [ 2.4  Tính hàm chi phí](#2.4)
    - [ Bài tập 1](#ex01)
  - [ 2.5 Gradient descent ](#2.5)
    - [ Bài tập 2](#ex02)
  - [ 2.6 Học tham số bằng batch gradient descent ](#2.6)


_**LƯU Ý:** Để tránh lỗi từ hệ thống chấm điểm tự động (autograder), bạn không được phép chỉnh sửa hoặc xóa các ô không được chấm điểm trong notebook này. Vui lòng cũng không thêm bất kỳ ô mới nào. 
**Sau khi bạn đã hoàn thành bài tập này** và muốn thử nghiệm với bất kỳ đoạn mã không được chấm điểm nào, bạn có thể làm theo hướng dẫn ở cuối notebook này._

<a name="1"></a>
## 1 - Các gói thư viện 

Đầu tiên, hãy chạy ô lệnh bên dưới để import tất cả các gói thư viện mà bạn sẽ cần trong bài tập này.
- [numpy](https://www.numpy.org) là gói thư viện cơ bản để làm việc với ma trận trong Python.
- [matplotlib](https://matplotlib.org) là một thư viện nổi tiếng để vẽ đồ thị trong Python.
- ``utils.py`` chứa các hàm hỗ trợ cho bài tập này. Bạn không cần chỉnh sửa mã trong tệp này.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
import copy
import math
%matplotlib inline

## 2 -  Phát biểu bài toán

Giả sử bạn là CEO của một chuỗi nhà hàng và đang xem xét các thành phố khác nhau để mở một chi nhánh mới.
- Bạn muốn mở rộng kinh doanh đến những thành phố có thể mang lại lợi nhuận cao hơn cho nhà hàng của bạn.
- Chuỗi nhà hàng đã có mặt tại nhiều thành phố khác nhau và bạn có dữ liệu về lợi nhuận và dân số của các thành phố đó.
- Bạn cũng có dữ liệu về các thành phố là ứng viên tiềm năng cho một nhà hàng mới. 
    - Đối với các thành phố này, bạn có dữ liệu về dân số thành phố.
    
Bạn có thể sử dụng dữ liệu này để giúp xác định thành phố nào có khả năng mang lại lợi nhuận cao hơn cho việc kinh doanh của bạn không?

## 3 - Tập dữ liệu

Bạn sẽ bắt đầu bằng cách nạp tập dữ liệu cho nhiệm vụ này. 
- Hàm `load_data()` được hiển thị bên dưới nạp dữ liệu vào các biến `x_train` và `y_train`
  - `x_train` là dân số của một thành phố
  - `y_train` là lợi nhuận của một nhà hàng ở thành phố đó. Giá trị âm cho lợi nhuận thể hiện một khoản lỗ.   
  - Cả `X_train` và `y_train` đều là mảng numpy.

In [ ]:
# load the dataset
x_train, y_train = load_data()

#### Xem các biến
Trước khi bắt đầu bất kỳ nhiệm vụ nào, việc làm quen thêm với tập dữ liệu của bạn rất hữu ích.  
- Một điểm khởi đầu tốt là chỉ cần in ra từng biến và xem nó chứa gì.

Đoạn mã bên dưới in ra biến `x_train` và kiểu của biến đó.

In [ ]:
# print x_train
print("Type of x_train:",type(x_train))
print("First five elements of x_train are:\n", x_train[:5]) 

`x_train` là một mảng numpy chứa các giá trị thập phân đều lớn hơn 0.
- Các giá trị này thể hiện dân số của thành phố nhân với 10.000
- Ví dụ, 6.1101 nghĩa là dân số của thành phố đó là 61.101
  
Bây giờ, hãy in `y_train`

In [ ]:
# print y_train
print("Type of y_train:",type(y_train))
print("First five elements of y_train are:\n", y_train[:5])  

Tương tự, `y_train` là một mảng numpy chứa các giá trị thập phân, một số âm, một số dương.
- Các giá trị này thể hiện lợi nhuận trung bình hàng tháng của nhà hàng bạn tại mỗi thành phố, tính theo đơn vị \$10.000.
  - Ví dụ, 17.592 thể hiện \$175.920 lợi nhuận trung bình hàng tháng cho thành phố đó.
  - -2.6807 thể hiện khoản lỗ trung bình hàng tháng -\$26.807 cho thành phố đó.

#### Kiểm tra kích thước của các biến

Một cách hữu ích khác để làm quen với dữ liệu của bạn là xem kích thước (dimensions) của nó.

Vui lòng in ra shape của `x_train` và `y_train` và xem có bao nhiêu ví dụ huấn luyện trong tập dữ liệu của bạn.

In [ ]:
print ('The shape of x_train is:', x_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(x_train))

Mảng dân số thành phố có 97 điểm dữ liệu, và lợi nhuận trung bình hàng tháng cũng có 97 điểm dữ liệu. Đây đều là các mảng NumPy 1 chiều.

#### Trực quan hóa dữ liệu của bạn

Việc hiểu dữ liệu thông qua trực quan hóa thường rất hữu ích. 
- Với tập dữ liệu này, bạn có thể dùng biểu đồ phân tán (scatter plot) để trực quan hóa dữ liệu, vì nó chỉ có hai thuộc tính để vẽ (lợi nhuận và dân số). 
- Nhiều bài toán khác mà bạn sẽ gặp trong thực tế có nhiều hơn hai thuộc tính (ví dụ, dân số, thu nhập hộ gia đình trung bình, lợi nhuận hàng tháng, doanh số hàng tháng). Khi bạn có nhiều hơn hai thuộc tính, bạn vẫn có thể dùng biểu đồ phân tán để xem mối quan hệ giữa từng cặp thuộc tính.


In [ ]:
# Create a scatter plot of the data. To change the markers to red "x",
# we used the 'marker' and 'c' parameters
plt.scatter(x_train, y_train, marker='x', c='r') 

# Set the title
plt.title("Profits vs. Population per city")
# Set the y-axis label
plt.ylabel('Profit in $10,000')
# Set the x-axis label
plt.xlabel('Population of City in 10,000s')
plt.show()

Mục tiêu của bạn là xây dựng một mô hình hồi quy tuyến tính để khớp với dữ liệu này.
- Với mô hình này, bạn sau đó có thể nhập vào dân số của một thành phố mới, và để mô hình ước tính lợi nhuận hàng tháng tiềm năng cho nhà hàng của bạn ở thành phố đó.

<a name="4"></a>
## 4 - Ôn lại về hồi quy tuyến tính

Trong bài lab thực hành này, bạn sẽ khớp các tham số hồi quy tuyến tính $(w,b)$ với tập dữ liệu của bạn.
- Hàm mô hình cho hồi quy tuyến tính, là hàm ánh xạ từ `x` (dân số thành phố) đến `y` (lợi nhuận hàng tháng của nhà hàng bạn tại thành phố đó) được biểu diễn như sau 
    $$f_{w,b}(x) = wx + b$$
    

- Để huấn luyện một mô hình hồi quy tuyến tính, bạn muốn tìm các tham số $(w,b)$ tốt nhất khớp với tập dữ liệu của bạn.  

    - Để so sánh xem một lựa chọn $(w,b)$ tốt hơn hay kém hơn lựa chọn khác, bạn có thể đánh giá nó bằng một hàm chi phí $J(w,b)$
      - $J$ là một hàm của $(w,b)$. Nghĩa là, giá trị chi phí $J(w,b)$ phụ thuộc vào giá trị của $(w,b)$.
  
    - Lựa chọn $(w,b)$ khớp tốt nhất với dữ liệu của bạn là lựa chọn có chi phí $J(w,b)$ nhỏ nhất.


- Để tìm các giá trị $(w,b)$ đạt được chi phí $J(w,b)$ nhỏ nhất có thể, bạn có thể sử dụng một phương pháp gọi là **gradient descent**. 
  - Với mỗi bước của gradient descent, các tham số $(w,b)$ của bạn sẽ tiến gần hơn đến các giá trị tối ưu đạt được chi phí $J(w,b)$ thấp nhất.
  

- Mô hình hồi quy tuyến tính đã huấn luyện sau đó có thể nhận đặc trưng đầu vào $x$ (dân số thành phố) và đưa ra một dự đoán $f_{w,b}(x)$ (lợi nhuận hàng tháng dự đoán cho một nhà hàng ở thành phố đó).

<a name="5"></a>
## 5 - Tính hàm chi phí

Gradient descent bao gồm các bước lặp đi lặp lại để điều chỉnh giá trị của tham số $(w,b)$ nhằm giảm dần chi phí $J(w,b)$.
- Ở mỗi bước của gradient descent, sẽ hữu ích cho bạn khi theo dõi tiến trình bằng cách tính chi phí $J(w,b)$ khi $(w,b)$ được cập nhật. 
- Trong phần này, bạn sẽ triển khai một hàm để tính $J(w,b)$ để có thể kiểm tra tiến trình của việc triển khai gradient descent.

#### Hàm chi phí
Như bạn có thể nhớ từ bài giảng, với một biến, hàm chi phí cho hồi quy tuyến tính $J(w,b)$ được định nghĩa là

$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$ 

- Bạn có thể xem $f_{w,b}(x^{(i)})$ là dự đoán của mô hình về lợi nhuận nhà hàng của bạn, trái ngược với $y^{(i)}$, là lợi nhuận thực tế được ghi lại trong dữ liệu.
- $m$ là số lượng ví dụ huấn luyện trong tập dữ liệu

#### Dự đoán của mô hình

- Đối với hồi quy tuyến tính với một biến, dự đoán của mô hình $f_{w,b}$ cho một ví dụ $x^{(i)}$ được biểu diễn như sau:

$$ f_{w,b}(x^{(i)}) = wx^{(i)} + b$$

Đây là phương trình của một đường thẳng, với hệ số chặn (intercept) $b$ và độ dốc (slope) $w$

#### Triển khai

Vui lòng hoàn thiện hàm `compute_cost()` bên dưới để tính chi phí $J(w,b)$.

<a name="ex01"></a>
### Bài tập 1

Hoàn thiện hàm `compute_cost` bên dưới để:

* Lặp qua các ví dụ huấn luyện, và với mỗi ví dụ, tính:
    * Dự đoán của mô hình cho ví dụ đó 
    $$
    f_{wb}(x^{(i)}) =  wx^{(i)} + b 
    $$
   
    * Chi phí cho ví dụ đó  $$cost^{(i)} =  (f_{wb} - y^{(i)})^2$$
    

* Trả về tổng chi phí trên tất cả các ví dụ
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} cost^{(i)}$$
  * Ở đây, $m$ là số lượng ví dụ huấn luyện và $\sum$ là toán tử tổng

Nếu bạn bị mắc kẹt, bạn có thể xem các gợi ý được trình bày sau ô lệnh bên dưới để giúp bạn triển khai.

In [ ]:
# UNQ_C1
# GRADED FUNCTION: compute_cost

def compute_cost(x, y, w, b): 
    """
    Computes the cost function for linear regression.
    
    Args:
        x (ndarray): Shape (m,) Input to the model (Population of cities) 
        y (ndarray): Shape (m,) Label (Actual profits for the cities)
        w, b (scalar): Parameters of the model
    
    Returns
        total_cost (float): The cost of using w,b as the parameters for linear regression
               to fit the data points in x and y
    """
    # number of training examples
    m = x.shape[0] 
    
    # You need to return this variable correctly
    total_cost = 0
    
    ### START CODE HERE ###
    
    ### END CODE HERE ### 

    return total_cost

<details>
  <summary><font size="3" color="darkgreen"><b>Nhấp để xem gợi ý</b></font></summary>
    
    
   * Bạn có thể biểu diễn một toán tử tổng, ví dụ: $h = \sum\limits_{i = 0}^{m-1} 2i$ trong mã như sau:
    
    ```python 
    h = 0
    for i in range(m):
        h = h + 2*i
    ```
  
   * Trong trường hợp này, bạn có thể lặp qua tất cả các ví dụ trong `x` bằng vòng lặp for và cộng dồn `cost` từ mỗi vòng lặp vào một biến (`cost_sum`) được khởi tạo bên ngoài vòng lặp.

   * Sau đó, bạn có thể trả về `total_cost` bằng `cost_sum` chia cho `2m`.
   * Nếu bạn mới học Python, hãy kiểm tra xem mã của bạn có được thụt lề đúng cách với dấu cách hoặc tab nhất quán hay không. Nếu không, nó có thể tạo ra kết quả khác hoặc gây ra lỗi `IndentationError: unexpected indent`. Bạn có thể tham khảo [chủ đề này](https://community.deeplearning.ai/t/indentation-in-python-indentationerror-unexpected-indent/159398) trong cộng đồng của chúng tôi để biết thêm chi tiết.

    <details>
          <summary><font size="2" color="darkblue"><b> Nhấp để xem thêm gợi ý</b></font></summary>
        
    * Đây là cách bạn có thể cấu trúc việc triển khai tổng thể cho hàm này
    
    ```python 
    def compute_cost(x, y, w, b):
        # number of training examples
        m = x.shape[0] 
    
        # You need to return this variable correctly
        total_cost = 0
    
        ### START CODE HERE ###  
        # Variable to keep track of sum of cost from each example
        cost_sum = 0
    
        # Loop over training examples
        for i in range(m):
            # Your code here to get the prediction f_wb for the ith example
            f_wb = 
            # Your code here to get the cost associated with the ith example
            cost = 
        
            # Add to sum of cost for each example
            cost_sum = cost_sum + cost 

        # Get the total cost as the sum divided by (2*m)
        total_cost = (1 / (2 * m)) * cost_sum
        ### END CODE HERE ### 

        return total_cost
    ```
    
    * Nếu bạn vẫn bị mắc kẹt, bạn có thể xem các gợi ý bên dưới để tìm hiểu cách tính `f_wb` và `cost`.
    
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý để tính f_wb</b></font></summary>
           &emsp; &emsp; Với các số vô hướng $a$, $b$ và $c$ (<code>x[i]</code>, <code>w</code> và <code>b</code> đều là số vô hướng), bạn có thể tính phương trình $h = ab + c$ trong mã như <code>h = a * b + c</code>
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; Thêm gợi ý để tính f</b></font></summary>
               &emsp; &emsp; Bạn có thể tính f_wb như sau <code>f_wb = w * x[i] + b </code>
           </details>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý để tính cost</b></font></summary>
          &emsp; &emsp; Bạn có thể tính bình phương của một biến z là z**2
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; Thêm gợi ý để tính cost</b></font></summary>
              &emsp; &emsp; Bạn có thể tính cost như sau <code>cost = (f_wb - y[i]) ** 2</code>
          </details>
    </details>
        
    </details>

</details>

    


Bạn có thể kiểm tra xem việc triển khai của mình có đúng không bằng cách chạy đoạn mã kiểm thử sau:

In [ ]:
# Compute cost with some initial values for paramaters w, b
initial_w = 2
initial_b = 1

cost = compute_cost(x_train, y_train, initial_w, initial_b)
print(type(cost))
print(f'Cost at initial w: {cost:.3f}')

# Public tests
from public_tests import *
compute_cost_test(compute_cost)

**Kết quả mong đợi**:
<table>
  <tr>
    <td> <b>Chi phí tại w ban đầu:<b> 75.203 </td> 
  </tr>
</table>

<a name="6"></a>
## 6 - Gradient descent 

Trong phần này, bạn sẽ triển khai gradient cho các tham số $w, b$ của hồi quy tuyến tính. 

Như đã mô tả trong các video bài giảng, thuật toán gradient descent là:

$$\begin{align*}& \text{lặp lại đến khi hội tụ:} \; \lbrace \newline \; & \phantom {0000} b := b -  \alpha \frac{\partial J(w,b)}{\partial b} \newline       \; & \phantom {0000} w := w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{1}  \; & 
\newline & \rbrace\end{align*}$$

trong đó, các tham số $w, b$ đều được cập nhật đồng thời và  
$$
\frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{2}
$$
$$
\frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) -y^{(i)})x^{(i)} \tag{3}
$$
* m là số lượng ví dụ huấn luyện trong tập dữ liệu

    
*  $f_{w,b}(x^{(i)})$ là dự đoán của mô hình, còn $y^{(i)}$, là giá trị mục tiêu


Bạn sẽ triển khai một hàm gọi là `compute_gradient` để tính $\frac{\partial J(w)}{\partial w}$, $\frac{\partial J(w)}{\partial b}$ 

<a name="ex02"></a>
### Bài tập 2

Vui lòng hoàn thiện hàm `compute_gradient` để:

* Lặp qua các ví dụ huấn luyện, và với mỗi ví dụ, tính:
    * Dự đoán của mô hình cho ví dụ đó 
    $$
    f_{wb}(x^{(i)}) =  wx^{(i)} + b 
    $$
   
    * Gradient cho các tham số $w, b$ từ ví dụ đó 
        $$
        \frac{\partial J(w,b)}{\partial b}^{(i)}  =  (f_{w,b}(x^{(i)}) - y^{(i)}) 
        $$
        $$
        \frac{\partial J(w,b)}{\partial w}^{(i)}  =  (f_{w,b}(x^{(i)}) -y^{(i)})x^{(i)} 
        $$
    

* Trả về tổng cập nhật gradient từ tất cả các ví dụ
    $$
    \frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} \frac{\partial J(w,b)}{\partial b}^{(i)}
    $$
    
    $$
    \frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} \frac{\partial J(w,b)}{\partial w}^{(i)} 
    $$
  * Ở đây, $m$ là số lượng ví dụ huấn luyện và $\sum$ là toán tử tổng

Nếu bạn bị mắc kẹt, bạn có thể xem các gợi ý được trình bày sau ô lệnh bên dưới để giúp bạn triển khai.

In [ ]:
# UNQ_C2
# GRADED FUNCTION: compute_gradient
def compute_gradient(x, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      x (ndarray): Shape (m,) Input to the model (Population of cities) 
      y (ndarray): Shape (m,) Label (Actual profits for the cities)
      w, b (scalar): Parameters of the model  
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
     """
    
    # Number of training examples
    m = x.shape[0]
    
    # You need to return the following variables correctly
    dj_dw = 0
    dj_db = 0
    
    ### START CODE HERE ###
    
    ### END CODE HERE ### 
        
    return dj_dw, dj_db

<details>
  <summary><font size="3" color="darkgreen"><b>Nhấp để xem gợi ý</b></font></summary>
    
   * Bạn có thể biểu diễn một toán tử tổng, ví dụ: $h = \sum\limits_{i = 0}^{m-1} 2i$ trong mã như sau:
    
   ```python 
    h = 0
    for i in range(m):
        h = h + 2*i
   ```
    
   * Trong trường hợp này, bạn có thể lặp qua tất cả các ví dụ trong `x` bằng vòng lặp for và với mỗi ví dụ, cộng dồn gradient từ ví dụ đó vào các biến `dj_dw` và `dj_db` được khởi tạo bên ngoài vòng lặp. 

   * Sau đó, bạn có thể trả về `dj_dw` và `dj_db` đều được chia cho `m`.    
    <details>
          <summary><font size="2" color="darkblue"><b> Nhấp để xem thêm gợi ý</b></font></summary>
        
    * Đây là cách bạn có thể cấu trúc việc triển khai tổng thể cho hàm này
    
    ```python 
    def compute_gradient(x, y, w, b): 
        """
        Computes the gradient for linear regression 
        Args:
          x (ndarray): Shape (m,) Input to the model (Population of cities) 
          y (ndarray): Shape (m,) Label (Actual profits for the cities)
          w, b (scalar): Parameters of the model  
        Returns
          dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
          dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
        """
    
        # Number of training examples
        m = x.shape[0]
    
        # You need to return the following variables correctly
        dj_dw = 0
        dj_db = 0
    
        ### START CODE HERE ### 
        # Loop over examples
        for i in range(m):  
            # Your code here to get prediction f_wb for the ith example
            f_wb = 
            
            # Your code here to get the gradient for w from the ith example 
            dj_dw_i = 
        
            # Your code here to get the gradient for b from the ith example 
            dj_db_i = 
     
            # Update dj_db : In Python, a += 1  is the same as a = a + 1
            dj_db += dj_db_i
        
            # Update dj_dw
            dj_dw += dj_dw_i
    
        # Divide both dj_dw and dj_db by m
        dj_dw = dj_dw / m
        dj_db = dj_db / m
        ### END CODE HERE ### 
        
        return dj_dw, dj_db
    ```
        
    * Nếu bạn vẫn bị mắc kẹt, bạn có thể xem các gợi ý bên dưới để tìm hiểu cách tính `f_wb` và `cost`.
    
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý để tính f_wb</b></font></summary>
           &emsp; &emsp; Bạn đã làm việc này ở bài tập trước! Với các số vô hướng $a$, $b$ và $c$ (<code>x[i]</code>, <code>w</code> và <code>b</code> đều là số vô hướng), bạn có thể tính phương trình $h = ab + c$ trong mã như <code>h = a * b + c</code>
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; Thêm gợi ý để tính f</b></font></summary>
               &emsp; &emsp; Bạn có thể tính f_wb như sau <code>f_wb = w * x[i] + b </code>
           </details>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý để tính dj_dw_i</b></font></summary>
           &emsp; &emsp; Với các số vô hướng $a$, $b$ và $c$ (<code>f_wb</code>, <code>y[i]</code> và <code>x[i]</code> đều là số vô hướng), bạn có thể tính phương trình $h = (a - b)c$ trong mã như <code>h = (a-b)*c</code>
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; Thêm gợi ý để tính f</b></font></summary>
               &emsp; &emsp; Bạn có thể tính dj_dw_i như sau <code>dj_dw_i = (f_wb - y[i]) * x[i] </code>
           </details>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý để tính dj_db_i</b></font></summary>
             &emsp; &emsp; Bạn có thể tính dj_db_i như sau <code> dj_db_i = f_wb - y[i] </code>
    </details>
        
    </details>

</details>

    


Chạy các ô lệnh bên dưới để kiểm tra việc triển khai hàm `compute_gradient` của bạn với hai cách khởi tạo tham số $w$,$b$ khác nhau.

In [ ]:
# Compute and display gradient with w initialized to zeroes
initial_w = 0
initial_b = 0

tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, initial_w, initial_b)
print('Gradient at initial w, b (zeros):', tmp_dj_dw, tmp_dj_db)

compute_gradient_test(compute_gradient)

Bây giờ hãy chạy thuật toán gradient descent đã triển khai ở trên trên tập dữ liệu của chúng ta.

**Kết quả mong đợi**:
<table>
  <tr>
    <td> <b>Gradient tại w, b ban đầu (bằng 0)<b></td>
    <td> -65.32884975 -5.83913505154639</td> 
  </tr>
</table>

In [ ]:
# Compute and display cost and gradient with non-zero w
test_w = 0.2
test_b = 0.2
tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, test_w, test_b)

print('Gradient at test w, b:', tmp_dj_dw, tmp_dj_db)

**Kết quả mong đợi**:
<table>
  <tr>
    <td> <b>Gradient tại w kiểm thử<b></td>
    <td> -47.41610118 -4.007175051546391</td> 
  </tr>
</table>

<a name="2.6"></a>
### 2.6 Học tham số bằng batch gradient descent 

Bây giờ bạn sẽ tìm các tham số tối ưu của một mô hình hồi quy tuyến tính bằng cách sử dụng batch gradient descent. Hãy nhớ rằng batch nghĩa là chạy tất cả các ví dụ trong một vòng lặp.
- Bạn không cần triển khai bất cứ điều gì cho phần này. Chỉ cần chạy các ô lệnh bên dưới. 

- Một cách tốt để kiểm tra xem gradient descent có hoạt động đúng hay không là xem
giá trị của $J(w,b)$ và kiểm tra xem nó có đang giảm dần ở mỗi bước hay không. 

- Giả sử bạn đã triển khai đúng gradient và tính chi phí chính xác, và bạn có một giá trị tốc độ học alpha phù hợp, $J(w,b)$ không bao giờ được tăng lên và nên hội tụ về một giá trị ổn định vào cuối thuật toán.

In [ ]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn theta. Updates theta by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      x :    (ndarray): Shape (m,)
      y :    (ndarray): Shape (m,)
      w_in, b_in : (scalar) Initial values of parameters of the model
      cost_function: function to compute cost
      gradient_function: function to compute the gradient
      alpha : (float) Learning rate
      num_iters : (int) number of iterations to run gradient descent
    Returns
      w : (ndarray): Shape (1,) Updated values of parameters of the model after
          running gradient descent
      b : (scalar)                Updated value of parameter of the model after
          running gradient descent
    """
    
    # number of training examples
    m = len(x)
    
    # An array to store cost J and w's at each iteration — primarily for graphing later
    J_history = []
    w_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_dw, dj_db = gradient_function(x, y, w, b )  

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            cost =  cost_function(x, y, w, b)
            J_history.append(cost)

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters/10) == 0:
            w_history.append(w)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
        
    return w, b, J_history, w_history #return w and J,w history for graphing

Bây giờ hãy chạy thuật toán gradient descent ở trên để học các tham số cho tập dữ liệu của chúng ta.

In [ ]:
# initialize fitting parameters. Recall that the shape of w is (n,)
initial_w = 0.
initial_b = 0.

# some gradient descent settings
iterations = 1500
alpha = 0.01

w,b,_,_ = gradient_descent(x_train ,y_train, initial_w, initial_b, 
                     compute_cost, compute_gradient, alpha, iterations)
print("w,b found by gradient descent:", w, b)

**Kết quả mong đợi**:
<table>
  <tr>
    <td> <b> w, b tìm được bằng gradient descent<b></td>
    <td> 1.16636235 -3.63029143940436</td> 
  </tr>
</table>

Bây giờ chúng ta sẽ sử dụng các tham số cuối cùng từ gradient descent để vẽ đường khớp tuyến tính. 

Hãy nhớ rằng chúng ta có thể lấy dự đoán cho một ví dụ đơn $f(x^{(i)})= wx^{(i)}+b$. 

Để tính các dự đoán trên toàn bộ tập dữ liệu, chúng ta có thể lặp qua tất cả các ví dụ huấn luyện và tính dự đoán cho từng ví dụ. Điều này được thể hiện trong khối mã bên dưới.

In [ ]:
m = x_train.shape[0]
predicted = np.zeros(m)

for i in range(m):
    predicted[i] = w * x_train[i] + b

Bây giờ chúng ta sẽ vẽ đồ thị các giá trị đã dự đoán để xem đường khớp tuyến tính.

In [ ]:
# Plot the linear fit
plt.plot(x_train, predicted, c = "b")

# Create a scatter plot of the data. 
plt.scatter(x_train, y_train, marker='x', c='r') 

# Set the title
plt.title("Profits vs. Population per city")
# Set the y-axis label
plt.ylabel('Profit in $10,000')
# Set the x-axis label
plt.xlabel('Population of City in 10,000s')

Các giá trị cuối cùng của $w,b$ cũng có thể được sử dụng để dự đoán lợi nhuận. Hãy dự đoán lợi nhuận sẽ như thế nào ở các khu vực có 35.000 và 70.000 người. 

- Mô hình nhận đầu vào là dân số của một thành phố tính theo đơn vị 10.000. 

- Do đó, 35.000 người có thể được chuyển thành đầu vào cho mô hình là `np.array([3.5])`

- Tương tự, 70.000 người có thể được chuyển thành đầu vào cho mô hình là `np.array([7.])`


In [ ]:
predict1 = 3.5 * w + b
print('For population = 35,000, we predict a profit of $%.2f' % (predict1*10000))

predict2 = 7.0 * w + b
print('For population = 70,000, we predict a profit of $%.2f' % (predict2*10000))

**Kết quả mong đợi**:
<table>
  <tr>
    <td> <b> Với dân số = 35.000, chúng ta dự đoán lợi nhuận là<b></td>
    <td> $4519.77 </td> 
  </tr>
  
  <tr>
    <td> <b> Với dân số = 70.000, chúng ta dự đoán lợi nhuận là<b></td>
    <td> $45342.45 </td> 
  </tr>
</table>

**Chúc mừng bạn đã hoàn thành bài lab thực hành này về hồi quy tuyến tính! Tuần tới, bạn sẽ xây dựng các mô hình để giải quyết một loại bài toán khác: phân loại (classification). Hẹn gặp lại ở đó!**

<details>
  <summary><font size="2" color="darkgreen"><b>Vui lòng nhấp vào đây nếu bạn muốn thử nghiệm với bất kỳ đoạn mã không được chấm điểm nào.</b></font></summary>
    <p><i><b>Lưu ý quan trọng: Vui lòng chỉ làm điều này sau khi bạn đã hoàn thành bài tập để tránh gặp vấn đề với hệ thống chấm điểm tự động.</b></i>
    <ol>
        <li> Trên menu của notebook, nhấp vào “View” > “Cell Toolbar” > “Edit Metadata”</li>
        <li> Nhấn nút “Edit Metadata” bên cạnh ô lệnh mà bạn muốn khóa/mở khóa</li>
        <li> Đặt giá trị thuộc tính cho “editable” thành:
            <ul>
                <li> “true” nếu bạn muốn mở khóa nó </li>
                <li> “false” nếu bạn muốn khóa nó </li>
            </ul>
        </li>
        <li> Trên menu của notebook, nhấp vào “View” > “Cell Toolbar” > “None” </li>
    </ol>
    <p> Đây là một đoạn demo ngắn về cách thực hiện các bước trên: 
        <br>
        <img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>